# Map the shapes of your cells

You have a table with one row per cell and a column for every shape measurement.
This notebook turns that table into a **shape map**: cells with similar shapes
end up close together, and cells are sorted into **shape clusters**.

What you do:

1. Point the notebook at your table (one cell below).
2. Run the cells from top to bottom.

What you get, in the output folder:

| File | What it holds |
|---|---|
| `shape_embedding.csv` | Your table plus the principal components (`PC1`...), the 2D map (`umap_1`, `umap_2`) and one cluster column per resolution (`leiden_r1.0`, ...) |
| `pca_loadings.csv` | Which measurements drive each principal component |
| `cluster_shape_stats.csv` | The median of every measurement in every cluster |
| `representative_cells.csv` | The most typical cells of each cluster |
| `analysis_config.json` | The settings you used, so you can repeat the run |
| `figures/` | The plots, as SVG (editable) and PNG |

The next notebook, *Group cells by their neighbourhood*, continues from `shape_embedding.csv`.

**Your table.** It is the file written at the end of the segmentation tutorial
(`all_stacks_features.csv`) or the merged table of the plugin's Export window
(`features_<date>_<n>stacks.csv`, in `artifacts/step4/exports/` of the
workspace). It needs the columns `stack_name` and `label`. Position columns
(`centroid-0`, `centroid-1`, `centroid-2`) are used for a few extra plots when
they exist. What each measurement means is listed in [FEATURES.md](FEATURES.md).

In [ ]:
# Make the helper file next to this notebook importable (works from the repo root or from this folder).
import sys
from pathlib import Path

for _p in (Path.cwd(), Path.cwd() / "tutorials" / "analysis", Path.cwd().parent):
    if (_p / "_helpers.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError("Could not find _helpers.py. Open this notebook from the repository root or from tutorials/analysis.")

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _helpers as h

warnings.filterwarnings("ignore", category=FutureWarning)
print("Ready.")

## 1. Edit this cell

This is the only cell you must change. Output folders are created in the folder you started Jupyter in, unless you give a full path.

- `TABLE_CSV`: your feature table.
- `METADATA_CSV`: optional. A small table with one row per stack: a `stack_name` column plus any columns you like (sample, region, treatment, age group...). Use `None` if you have none.
- `FEATURES`: the measurements that define "shape". `None` uses the 31 standard shape measurements that your table has. Or give your own list of column names.
- `COLOR_BY`: extra columns to colour the map by. They can come from the table or from the metadata file.
- `LABELS_TIFF`, `MESHES_DIR`: optional. Only needed if you want pictures of the typical cells (see section 8).
- `MAX_CELLS_FOR_MAP`: for very large tables, set a number such as `200000` to draw the 2D map (UMAP) from a random subset of cells. Everything else (principal components, clusters, the saved table) always uses every cell; the cells left out of the map have empty `umap_1` / `umap_2`. `None` maps every cell.
- `LOG_RULE`: which measurements get a logarithm before PCA (section 3). `"original"` logs every never-negative measurement that is strongly skewed to either side. `"workbench"` follows the plugin's analysis workbench: measurements with a natural 0-to-1 range (sphericity, solidity...) are never logged, and only right-skewed ones are.
- `N_PCS`, `USE_SIZE_CORRECTION`, `WINSOR_CLIP`, `KNN`, `LEIDEN_RESOLUTIONS`, `ACTIVE_RES`, `UMAP_NEIGHBORS`, `UMAP_MIN_DIST`, `RUN_TSNE`, `METRIC`, `RANDOM_SEED`, `N_REPS`, `MAX_STACK_PANELS`, `VIOLIN_FEATURES`, `SAVE_FIGURES`: defaults that work for most tables. How to choose `N_PCS`, `KNN`, `LEIDEN_RESOLUTIONS` and `USE_SIZE_CORRECTION` is explained where they are used.

In [ ]:
# ---------------- EDIT ME ----------------
TABLE_CSV = "all_stacks_features.csv"
METADATA_CSV = None            # for example "metadata.csv"
OUTPUT_DIR = "shape_output"

FEATURES = None                # None = the standard shape measurements your table has
COLOR_BY = []                  # for example ["region", "sample"]

LABELS_TIFF = None             # optional: labels volume of ONE stack, to draw typical cells
LABELS_STACK_NAME = None       # the stack_name that LABELS_TIFF belongs to (None if the table has one stack)
VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)   # size of one voxel (z, y, x), in the units you want on the drawings
MESHES_DIR = None              # optional: the plugin's mesh folder, <MESHES_DIR>/<stack_name>/cell_<label>.obj

MAX_CELLS_FOR_MAP = None       # None = map every cell; a number = draw the map from a random subset

# ---------------- Settings with defaults ----------------
LOG_RULE = "original"          # "original" or "workbench" (see section 1)
N_PCS = 10                     # how many principal components to keep
USE_SIZE_CORRECTION = False    # True = remove what is just cell size, so clusters reflect shape only
WINSOR_CLIP = 5.0              # clip values further than this many inter-quartile ranges from the median
KNN = 15                       # neighbours per cell in the similarity graph
LEIDEN_RESOLUTIONS = [0.3, 0.5, 1.0, 1.5, 2.0]   # higher = more, smaller clusters
ACTIVE_RES = 1.0               # the resolution used for the plots below (must be in the list above)
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
RUN_TSNE = False               # True = also compute a t-SNE map (slower)
METRIC = "euclidean"
RANDOM_SEED = 42
N_REPS = 10                    # typical cells kept per cluster
MAX_STACK_PANELS = 12          # most stacks drawn in the per-stack figure
VIOLIN_FEATURES = ["centroid-0", "sphericity", "true_sphericity", "aspect_ratio", "z_elongation",
                   "solidity", "fill_ratio", "inertia_elongation"]   # measurements drawn as violins in section 7
SAVE_FIGURES = True

## 2. Load the table

The notebook keeps only the measurements your table has. A measurement that is
missing for more than 10% of the cells (for example one that was not measured in
some stack) is dropped first, so it does not remove whole stacks. Cells with a
missing value in any remaining measurement are then dropped. The notebook tells
you what it dropped.

**Orientation.** Some measurements depend on which side of the tissue is "up":
`apical_area`, `basal_area`, `apical_basal_area_ratio`,
`max_cross_section_z_relative`, `cross_section_taper`, `cross_section_bulge` and the
depth column `centroid-0`. The orientation is set when the segmentation is made
(in the segmentation tutorial or in the plugin). It must be the same for every stack in your table.
If some stacks are flipped, remove those measurements with `FEATURES`, or flip the stacks first.

In [ ]:
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir = out_dir / "figures"

df_raw = h.load_table(TABLE_CSV)
df_raw = h.attach_metadata(df_raw, METADATA_CSV)
print(f"Loaded {len(df_raw):,} cells from {df_raw['stack_name'].nunique()} stack(s).")

wanted = list(FEATURES) if FEATURES else h.TARGET_FEATURES
features = [f for f in wanted if f in df_raw.columns]
missing = [f for f in wanted if f not in df_raw.columns]
if missing:
    print(f"{len(missing)} measurement(s) are not in your table and are skipped: {missing}")
features = h.drop_sparse_features(df_raw, features, max_missing=0.10)
if len(features) < 3:
    raise ValueError("Fewer than 3 usable measurements. Check FEATURES against the column names of your table.")

df_raw[features] = df_raw[features].replace([np.inf, -np.inf], np.nan)
df = df_raw.dropna(subset=features).reset_index(drop=True)
print(f"{len(df_raw) - len(df):,} cell(s) dropped for missing values; {len(df):,} left. Using {len(features)} measurements.")
if len(df) < 50:
    raise ValueError(f"Only {len(df)} cells are left after dropping missing values. Check FEATURES and the table.")

## 3. Prepare the measurements and run PCA

Measurements have different units and very different ranges. Three steps make
them comparable:

1. Strongly skewed measurements (for example volume) are log-transformed, with `log(1 + value)`. `LOG_RULE = "original"` logs every never-negative measurement whose skew is strong on either side; `LOG_RULE = "workbench"` is the plugin's rule: 0-to-1 measurements are never logged and only right-skewed ones are.
2. Each measurement is centred on its median and divided by its spread. A few huge cells then do not dominate.
3. Extreme values are clipped.

**PCA** then combines the measurements into a few new axes (principal
components, PC1, PC2...). PC1 is the direction along which cells differ most.
The scree plot shows how much each PC explains. The loadings heatmap shows which
measurements make up each PC: strong red or blue means the measurement matters
for that PC.

**Choosing `N_PCS`.** Keep enough PCs to hold roughly 80 to 90% of the variance
(the printed "together" column), and no more than the scree plot needs: PCs after
the curve flattens mostly carry noise.

**Choosing `USE_SIZE_CORRECTION`.** Many shape measurements simply grow with cell size.
If the loadings show that PC1 is mostly size measurements and you want to compare shape
independently of size, switch it on: the part of each measurement that follows size is removed first.
Leave it off if size is part of what you want to separate.

In [ ]:
X, log_applied = h.preprocess_features(df[features].to_numpy(dtype=float), features, WINSOR_CLIP, log_rule=LOG_RULE)

if USE_SIZE_CORRECTION:
    X, used_features = h.residualise_size(X, features, h.SIZE_REGRESSORS, RANDOM_SEED)
else:
    used_features = features

pcs, loadings, variance = h.run_pca(X, used_features, N_PCS, RANDOM_SEED)
for i, v in enumerate(variance):
    print(f"PC{i + 1}: {v:.1%}   (together {variance[:i + 1].sum():.1%})")

fig = h.plot_scree_loadings(variance, loadings, max_pcs=N_PCS)
if SAVE_FIGURES:
    h.save_fig(fig, "pca_scree_loadings", fig_dir)
plt.show()

## 4. Find shape clusters

Each cell is linked to the `KNN` (default 15) cells with the most similar shape. Clusters are
groups of cells that are more linked to each other than to the rest
(Leiden clustering).

The **resolution** sets how fine the grouping is. Low resolution gives a few
big groups. High resolution gives many small ones. There is no single right
value, so the notebook runs several and you choose.

**How to choose `KNN` and the resolution.** `KNN` between 15 and 30 works for most tables;
a larger value gives smoother, larger clusters. For the resolution, look at the table of cluster
counts below: pick a resolution inside a plateau where the number of clusters changes slowly,
check that the typical cells and violins of neighbouring clusters really differ (sections 7 and 8),
and be wary of clusters with under about 1% of the cells.

In [ ]:
memberships = h.leiden_multi(pcs, LEIDEN_RESOLUTIONS, k=KNN, metric=METRIC, seed=RANDOM_SEED)
for res, m in memberships.items():
    df[h.res_name(res)] = m
print("\nClusters per resolution:")
print(pd.Series({float(r): len(set(m)) for r, m in memberships.items()}, name="clusters").rename_axis("resolution").to_string())

ACTIVE_COL = h.res_name(ACTIVE_RES)
if ACTIVE_COL not in df.columns:
    raise ValueError(f"ACTIVE_RES={float(ACTIVE_RES)} must be one of LEIDEN_RESOLUTIONS={[float(r) for r in LEIDEN_RESOLUTIONS]}")
palette = h.assign_palette(df[ACTIVE_COL])
print(f"\nShowing resolution {ACTIVE_RES}: {len(palette)} clusters.")
print(df[ACTIVE_COL].value_counts().sort_index().to_string())

## 5. Draw the map (UMAP)

UMAP squeezes the PCs into two dimensions so you can look at them. Cells close
together have similar shapes. Distances between far-apart groups are not
precise, so read the map as "who is next to whom", not as a ruler.

In [ ]:
if MAX_CELLS_FOR_MAP is not None and len(df) > MAX_CELLS_FOR_MAP:
    map_idx = np.sort(np.random.default_rng(RANDOM_SEED).choice(len(df), int(MAX_CELLS_FOR_MAP), replace=False))
    print(f"Drawing the map from a random subset of {len(map_idx):,} of {len(df):,} cells (the other cells have empty map columns).")
else:
    map_idx = np.arange(len(df))
um = h.run_umap(pcs[map_idx], UMAP_NEIGHBORS, UMAP_MIN_DIST, METRIC, RANDOM_SEED)
df["umap_1"], df["umap_2"] = np.nan, np.nan
df.loc[df.index[map_idx], "umap_1"], df.loc[df.index[map_idx], "umap_2"] = um[:, 0], um[:, 1]
if RUN_TSNE:
    ts = h.run_tsne(pcs[map_idx], metric=METRIC, seed=RANDOM_SEED)
    df["tsne_1"], df["tsne_2"] = np.nan, np.nan
    df.loc[df.index[map_idx], "tsne_1"], df.loc[df.index[map_idx], "tsne_2"] = ts[:, 0], ts[:, 1]
print("Map done.")

## 6. Look at the map

Each panel is the same map with a different colouring:

- the shape clusters,
- every column you listed in `COLOR_BY`,
- depth in the tissue (`centroid-0`), when your table has it,
- one shape measurement, so you can see what a region of the map means.

If your table holds several stacks, a second figure shows where each cluster
sits inside each stack.

In [ ]:
panels = [("Shape clusters (resolution %s)" % ACTIVE_RES, ACTIVE_COL, "cat")]
for col in COLOR_BY:
    if col in df.columns:
        panels.append((col, col, "cat" if h.column_kind(df[col]) == "categorical" else "num"))
    else:
        print(f"COLOR_BY column '{col}' is not in the table or metadata; skipped.")
if "centroid-0" in df.columns:
    panels.append(("Position along z (centroid-0)", "centroid-0", "num"))
feat_show = next((f for f in ("true_sphericity", "sphericity", "solidity", "aspect_ratio") if f in df.columns), used_features[0])
panels.append((feat_show, feat_show, "num"))

ncols = min(3, len(panels))
nrows = int(np.ceil(len(panels) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 5.5 * nrows), squeeze=False)
for ax, (title, col, kind) in zip(axes.ravel(), panels):
    if kind == "cat":
        pal = palette if col == ACTIVE_COL else h.assign_palette(df[col])
        h.scatter_by(ax, df["umap_1"], df["umap_2"], df[col], pal, title=title)
    else:
        h.scatter_continuous(ax, df["umap_1"], df["umap_2"], df[col], title=title, label=col)
for ax in axes.ravel()[len(panels):]:
    ax.set_visible(False)
fig.tight_layout()
if SAVE_FIGURES:
    h.save_fig(fig, "umap_overview", fig_dir)
plt.show()

In [ ]:
# Per-stack view: only when the table holds more than one stack and has positions.
stacks = sorted(df["stack_name"].unique())
if len(stacks) > 1 and {"centroid-1", "centroid-2"}.issubset(df.columns):
    shown = stacks[:MAX_STACK_PANELS]
    ncols = min(4, len(shown))
    nrows = int(np.ceil(len(shown) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
    for ax, name in zip(axes.ravel(), shown):
        sub = df[df["stack_name"] == name]
        h.scatter_by(ax, sub["centroid-2"], sub["centroid-1"], sub[ACTIVE_COL], palette, s=4, alpha=0.8,
                     title=f"{name[:28]} (n={len(sub):,})", legend=False)
        ax.set_aspect("equal")
        ax.invert_yaxis()
    for ax in axes.ravel()[len(shown):]:
        ax.set_visible(False)
    fig.suptitle("Where each cluster sits inside each stack (top-down view)")
    fig.tight_layout()
    if SAVE_FIGURES:
        h.save_fig(fig, "clusters_per_stack", fig_dir)
    plt.show()
else:
    print("One stack (or no position columns): the per-stack figure is skipped.")

## 7. What does each cluster look like?

The violins show, for a handful of key measurements, how the values spread
inside each cluster. The table `cluster_shape_stats.csv` holds the median of
every measurement per cluster, for every resolution.

In [ ]:
violin_feats = [f for f in VIOLIN_FEATURES if f in df.columns]
if violin_feats:
    ids = sorted(df[ACTIVE_COL].unique())
    ncols = min(4, len(violin_feats))
    nrows = int(np.ceil(len(violin_feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.4 * nrows), squeeze=False)
    for ax, feat in zip(axes.ravel(), violin_feats):
        data = [df.loc[df[ACTIVE_COL] == c, feat].dropna().to_numpy() for c in ids]
        parts = ax.violinplot(data, positions=range(len(ids)), showmedians=True, showextrema=False)
        for body, c in zip(parts["bodies"], ids):
            body.set_facecolor(palette[c]); body.set_alpha(0.7)
        ax.set_xticks(range(len(ids)))
        ax.set_xticklabels([str(c) for c in ids], fontsize=6)
        ax.set_title(feat, fontsize=9)
        ax.set_xlabel("cluster", fontsize=7)
    for ax in axes.ravel()[len(violin_feats):]:
        ax.set_visible(False)
    fig.tight_layout()
    if SAVE_FIGURES:
        h.save_fig(fig, "cluster_violins", fig_dir)
    plt.show()

stats = pd.concat(
    [h.cluster_summary(df, h.res_name(r), used_features).assign(resolution=r) for r in LEIDEN_RESOLUTIONS],
    ignore_index=True,
)
stats = stats[["resolution"] + [c for c in stats.columns if c != "resolution"]]
stats[stats["resolution"] == ACTIVE_RES].head(10)

## 8. Typical cells of each cluster

A cluster is easier to understand when you look at its cells. For every cluster
the notebook picks the cells closest to the cluster's median in PC space: the
most typical ones. They are listed in `representative_cells.csv` (stack and
label), so you can find them in the segmentation.

Drawings are optional:

- Give `LABELS_TIFF` (the labels volume of one stack) and the cells of that stack are drawn from it.
- Or give `MESHES_DIR` (the plugin's mesh folder) and the saved meshes are used.

All cells are drawn at the same scale.

In [ ]:
reps = h.select_representatives(pcs, df, ACTIVE_COL, N_REPS)
display_reps = reps.head(10)
display_reps

In [ ]:
labels_volume = h.read_labels_volume(LABELS_TIFF) if LABELS_TIFF else None
labels_stack = LABELS_STACK_NAME or (stacks[0] if len(stacks) == 1 else None)

if labels_volume is None and MESHES_DIR is None:
    print("No LABELS_TIFF or MESHES_DIR given: drawings skipped. The table above lists the typical cells.")
else:
    fig = h.plot_cluster_gallery(reps, palette, labels_volume, labels_stack, VOXEL_SIZE_ZYX, MESHES_DIR)
    if fig is None:
        print("No mesh could be made for the typical cells (check the stack name and the folder).")
    else:
        if SAVE_FIGURES:
            h.save_fig(fig, "typical_cells", fig_dir)
        plt.show()

## 9. Save

Everything is written to the output folder.

In [ ]:
pc_names = [f"PC{i + 1}" for i in range(pcs.shape[1])]
result = pd.concat([df, pd.DataFrame(pcs, columns=pc_names)], axis=1)
front = [c for c in ("stack_name", "label") if c in result.columns]
result = result[front + [c for c in result.columns if c not in front]]

result.to_csv(out_dir / "shape_embedding.csv", index=False)
loadings.to_csv(out_dir / "pca_loadings.csv")
stats.to_csv(out_dir / "cluster_shape_stats.csv", index=False)
reps.to_csv(out_dir / "representative_cells.csv", index=False)

h.write_json(out_dir / "analysis_config.json", {
    "table": Path(TABLE_CSV).name,
    "n_cells": len(df),
    "n_stacks": len(stacks),
    "features_used": used_features,
    "log1p_applied": log_applied,
    "size_correction": USE_SIZE_CORRECTION,
    "n_pcs": pcs.shape[1],
    "variance_explained": variance,
    "log_rule": LOG_RULE,
    "winsor_clip": WINSOR_CLIP,
    "knn": KNN,
    "leiden_resolutions": LEIDEN_RESOLUTIONS,
    "active_resolution": ACTIVE_RES,
    "n_clusters": {str(r): int(len(set(m))) for r, m in memberships.items()},
    "umap_neighbors": UMAP_NEIGHBORS,
    "umap_min_dist": UMAP_MIN_DIST,
    "metric": METRIC,
    "random_seed": RANDOM_SEED,
    "max_cells_for_map": MAX_CELLS_FOR_MAP,
    "n_cells_in_map": int(len(map_idx)),
    "adjacency_dir": df_raw.attrs.get("adjacency_dir"),
})
print("Wrote:", ", ".join(sorted(p.name for p in out_dir.iterdir() if p.is_file())))